# Iterative Agent Development with Pydantic AI

This notebook demonstrates the **iterative development process** for building AI agents. Rather than showing a polished final product, we'll build a **Personal Task Manager** agent step-by-step, discovering limitations and improving our design through 4 iterations.

**What you'll learn:**
- How to identify agent limitations through testing
- Progressive enhancement of agent capabilities
- When to use tools vs. structured output vs. dependency injection
- Validation patterns for robust agent behavior

**Our development journey:**

| Iteration | Focus | Key Pattern |
|-----------|-------|-------------|
| V1 | Basic Todo Agent | `@agent.tool_plain` |
| V2 | Context-Aware | Dependency injection |
| V3 | Structured Analysis | `output_type` |
| V4 | Intelligent Validation | `@agent.output_validator` |

## Setup

First, we'll load environment variables and import what we need.

In [ ]:
# Load API keys from .env file if present
from dotenv import load_dotenv

load_dotenv("../.env")

from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Literal
import json

from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, ModelRetry

from pydantic_ai_jupyter import run_in_jupyter

---

## Iteration 1: Basic Todo Agent

Let's start with the simplest possible task manager. Our first version will:
- Add tasks to a list
- List all tasks
- Mark tasks as complete

We'll use a simple in-memory list and `@agent.tool_plain` decorators (no dependencies yet).

In [ ]:
# V1: Simple in-memory task storage (global variable)
tasks_v1: list[dict] = []

task_agent_v1 = Agent(
    "openai:gpt-4o-mini",
    system_prompt="""You are a helpful task manager assistant. 
Help users manage their todo list by adding tasks, listing them, and marking them complete.
Always confirm actions you take.""",
)


@task_agent_v1.tool_plain
def add_task(title: str, description: str = "") -> str:
    """Add a new task to the todo list."""
    task = {
        "id": len(tasks_v1) + 1,
        "title": title,
        "description": description,
        "completed": False,
    }
    tasks_v1.append(task)
    return f"Added task #{task['id']}: {title}"


@task_agent_v1.tool_plain
def list_tasks() -> str:
    """List all tasks in the todo list."""
    if not tasks_v1:
        return "No tasks found. Your todo list is empty!"

    lines = []
    for task in tasks_v1:
        status = "DONE" if task["completed"] else "TODO"
        lines.append(f"#{task['id']} [{status}] {task['title']}")
        if task["description"]:
            lines.append(f"   {task['description']}")
    return "\n".join(lines)


@task_agent_v1.tool_plain
def complete_task(task_id: int) -> str:
    """Mark a task as completed."""
    for task in tasks_v1:
        if task["id"] == task_id:
            task["completed"] = True
            return f"Marked task #{task_id} as complete: {task['title']}"
    return f"Task #{task_id} not found"

### Testing V1

Let's try adding some tasks and see how our agent behaves.

In [ ]:
# Clear tasks for a fresh start
tasks_v1.clear()

result = await run_in_jupyter(
    task_agent_v1,
    "Add a task to buy groceries and another to call mom",
)

In [ ]:
# Continue the conversation using message_history
result = await run_in_jupyter(
    task_agent_v1,
    "Show me my tasks",
    message_history=result.all_messages(),
)

In [ ]:
# Complete a task
result = await run_in_jupyter(
    task_agent_v1,
    "Mark the groceries task as done",
    message_history=result.all_messages(),
)

### V1 Limitations Discovered

Our basic agent works, but we've discovered some issues:

1. **Global State Problem**: The `tasks_v1` list is global - we can't support multiple users or test in isolation
2. **No User Context**: The agent doesn't know who it's talking to or their preferences
3. **No Priority/Due Dates**: All tasks are treated equally, no way to prioritize
4. **Plain Text Output**: We can only get string responses, making it hard to use data programmatically

**Next iteration goal:** Introduce dependency injection to manage state properly.

---

## Iteration 2: Context-Aware Agent with Dependency Injection

Now we'll refactor to use pydantic-ai's dependency injection. This gives us:
- Per-user task storage (no more global state)
- User preferences available to the agent
- Dynamic system prompts based on context
- Cleaner, more testable code

In [ ]:
@dataclass
class TaskStorage:
    """Storage for a user's tasks."""

    tasks: list[dict] = field(default_factory=list)

    def add(
        self,
        title: str,
        description: str = "",
        priority: str = "medium",
        due_date: str | None = None,
    ) -> dict:
        task = {
            "id": len(self.tasks) + 1,
            "title": title,
            "description": description,
            "priority": priority,
            "due_date": due_date,
            "completed": False,
            "created_at": datetime.now().isoformat(),
        }
        self.tasks.append(task)
        return task

    def get_pending(self) -> list[dict]:
        return [t for t in self.tasks if not t["completed"]]

    def get_by_id(self, task_id: int) -> dict | None:
        for task in self.tasks:
            if task["id"] == task_id:
                return task
        return None


@dataclass
class UserContext:
    """User context passed to the agent via dependency injection."""

    user_name: str
    storage: TaskStorage
    preferences: dict = field(
        default_factory=lambda: {
            "default_priority": "medium",
            "work_hours": "9am-5pm",
        }
    )

In [ ]:
task_agent_v2 = Agent(
    "openai:gpt-4o-mini",
    deps_type=UserContext,  # Declare the dependency type
    system_prompt="You are a personal task manager assistant.",
)


@task_agent_v2.system_prompt
def personalized_prompt(ctx: RunContext[UserContext]) -> str:
    """Dynamic system prompt based on user context."""
    pending = len(ctx.deps.storage.get_pending())
    return f"""You're helping {ctx.deps.user_name} manage their tasks.
They currently have {pending} pending task(s).
Their work hours are {ctx.deps.preferences['work_hours']}.
Default priority for new tasks: {ctx.deps.preferences['default_priority']}."""


@task_agent_v2.tool
def add_task(
    ctx: RunContext[UserContext],
    title: str,
    description: str = "",
    priority: Literal["low", "medium", "high"] = "medium",
    due_date: str | None = None,
) -> str:
    """Add a new task. Priority can be low, medium, or high. Due date format: YYYY-MM-DD."""
    task = ctx.deps.storage.add(title, description, priority, due_date)
    return f"Added task #{task['id']}: {title} (priority: {priority})"


@task_agent_v2.tool
def list_tasks(ctx: RunContext[UserContext], include_completed: bool = False) -> str:
    """List tasks. By default shows only pending tasks."""
    tasks = (
        ctx.deps.storage.tasks if include_completed else ctx.deps.storage.get_pending()
    )

    if not tasks:
        return f"No tasks found for {ctx.deps.user_name}!"

    lines = [f"Tasks for {ctx.deps.user_name}:"]
    for task in tasks:
        status = "DONE" if task["completed"] else task["priority"].upper()
        due = f" (due: {task['due_date']})" if task.get("due_date") else ""
        lines.append(f"#{task['id']} [{status}] {task['title']}{due}")
    return "\n".join(lines)


@task_agent_v2.tool
def complete_task(ctx: RunContext[UserContext], task_id: int) -> str:
    """Mark a task as completed."""
    task = ctx.deps.storage.get_by_id(task_id)
    if task:
        task["completed"] = True
        return f"Great job, {ctx.deps.user_name}! Completed: {task['title']}"
    return f"Task #{task_id} not found"

### Testing V2

Now we can create isolated contexts for different users. Notice how the agent responds differently based on who's asking.

In [ ]:
# Create Alice's context with her preferences
alice_storage = TaskStorage()
alice = UserContext(
    user_name="Alice",
    storage=alice_storage,
    preferences={"default_priority": "high", "work_hours": "8am-4pm"},
)

result = await run_in_jupyter(
    task_agent_v2,
    "Add these tasks: 1) Prepare presentation for tomorrow (high priority), "
    "2) Review team updates, 3) Book dentist appointment (low priority, due 2025-02-15)",
    deps=alice,
)

In [ ]:
# Create Bob's context - completely separate storage
bob_storage = TaskStorage()
bob = UserContext(user_name="Bob", storage=bob_storage)

result = await run_in_jupyter(
    task_agent_v2,
    "What tasks do I have?",
    deps=bob,
)

In [ ]:
# Verify isolation: Alice's tasks are separate from Bob's
print(f"Alice has {len(alice_storage.tasks)} tasks")
print(f"Bob has {len(bob_storage.tasks)} tasks")

### V2 Improvements and Remaining Limitations

**What we fixed:**
- Each user now has isolated storage
- Agent knows the user's name and preferences
- Tasks have priority and due dates
- Code is testable (we can inject mock storage)

**New limitations discovered:**

1. **Unstructured Analysis**: When asking "what should I focus on?", we get free-form text
2. **No Programmatic Access**: Can't easily extract priority rankings from the response
3. **Limited Reasoning**: Agent can't easily compare and rank tasks systematically

**Next iteration goal:** Add structured output for task analysis and recommendations.

---

## Iteration 3: Structured Output for Task Analysis

Now we'll add a specialized agent that returns **structured analysis**. This lets us:
- Get programmatic access to task insights
- Display rich analysis in our UI
- Build automated workflows based on agent recommendations

In [ ]:
class TaskRecommendation(BaseModel):
    """A single task recommendation with urgency analysis."""

    task_id: int = Field(description="The ID of the task")
    title: str = Field(description="The task title")
    urgency_score: float = Field(
        ge=0, le=10, description="How urgent this task is (0-10)"
    )
    reasoning: str = Field(description="Why this task has this urgency")


class DailyPlan(BaseModel):
    """A structured daily plan with prioritized tasks."""

    greeting: str = Field(description="Personalized greeting for the user")
    focus_tasks: list[TaskRecommendation] = Field(
        description="Top 3 tasks to focus on today, ordered by priority"
    )
    defer_tasks: list[str] = Field(
        description="Task titles that can wait until later"
    )
    productivity_tip: str = Field(
        description="A helpful productivity tip based on the task list"
    )
    estimated_work_hours: float = Field(
        ge=0, le=24, description="Estimated hours to complete focus tasks"
    )

In [ ]:
# Specialized agent for task analysis with structured output
analysis_agent = Agent(
    "openai:gpt-4o-mini",
    deps_type=UserContext,
    output_type=DailyPlan,  # Structured output!
    system_prompt="""You are a productivity coach analyzing task lists.
Evaluate tasks based on:
- Due dates (urgent if soon or overdue)
- Priority level set by user
- Task dependencies and context
Provide actionable, realistic plans.""",
)


@analysis_agent.system_prompt
def analysis_context(ctx: RunContext[UserContext]) -> str:
    """Provide current date and user context for analysis."""
    today = datetime.now().strftime("%Y-%m-%d")
    return f"""Today's date: {today}
User: {ctx.deps.user_name}
Work hours: {ctx.deps.preferences.get('work_hours', '9am-5pm')}"""


@analysis_agent.tool
def get_all_tasks(ctx: RunContext[UserContext]) -> str:
    """Get all tasks with full details for analysis."""
    tasks = ctx.deps.storage.tasks
    if not tasks:
        return "No tasks in the system."

    return json.dumps(tasks, indent=2, default=str)

### Testing V3

Let's create a realistic scenario with tasks at various priority levels and due dates.

In [ ]:
# Set up a user with several tasks for analysis
analysis_storage = TaskStorage()

# Add tasks with varying urgency
today = datetime.now()
analysis_storage.add(
    "Prepare quarterly report", "Q4 numbers and projections", "high",
    (today + timedelta(days=3)).strftime("%Y-%m-%d")
)
analysis_storage.add("Review pull requests", "3 PRs waiting", "medium")
analysis_storage.add("Team standup", "Daily 10am meeting", "medium",
    today.strftime("%Y-%m-%d")
)
analysis_storage.add(
    "Book flight for conference", "March conference", "low",
    (today + timedelta(days=45)).strftime("%Y-%m-%d")
)
analysis_storage.add(
    "Fix production bug", "Customer-reported issue", "high",
    today.strftime("%Y-%m-%d")
)
analysis_storage.add("Update documentation", "API docs outdated", "low")

analyst_user = UserContext(
    user_name="Developer",
    storage=analysis_storage,
    preferences={"work_hours": "9am-5pm"},
)

print(f"Created {len(analysis_storage.tasks)} tasks for analysis")

In [ ]:
result = await run_in_jupyter(
    analysis_agent,
    "What should I focus on today?",
    deps=analyst_user,
)

In [ ]:
# Now we have structured, programmatic access to the plan!
plan = result.output

print(f"Greeting: {plan.greeting}")
print(f"\nFocus Tasks ({plan.estimated_work_hours}h estimated):")
for rec in plan.focus_tasks:
    print(f"  #{rec.task_id} {rec.title} (urgency: {rec.urgency_score}/10)")
    print(f"     Reason: {rec.reasoning}")
print(f"\nCan defer: {', '.join(plan.defer_tasks)}")
print(f"\nTip: {plan.productivity_tip}")

In [ ]:
# We can now build workflows based on the structured output
high_urgency_tasks = [t for t in plan.focus_tasks if t.urgency_score >= 8]

if high_urgency_tasks:
    print(f"ALERT: You have {len(high_urgency_tasks)} high-urgency task(s)!")
    for task in high_urgency_tasks:
        print(f"  - {task.title}: {task.reasoning}")

### V3 Improvements and Remaining Limitations

**What we achieved:**
- Structured, type-safe output we can use programmatically
- Rich analysis with urgency scores and reasoning
- Separation of concerns: task management vs. task analysis agents

**New limitations discovered:**

1. **No Validation Logic**: Agent might suggest task IDs that don't exist
2. **No Date Sanity Checks**: Could suggest overdue tasks as "deferrable"
3. **Hallucination Risk**: Urgency scores are purely LLM-generated, not grounded in actual data

**Next iteration goal:** Add output validators to ensure recommendations are valid and realistic.

---

## Iteration 4: Intelligent Validation for Robust Recommendations

Our final iteration adds **output validation** to ensure the agent's recommendations are:
- Referencing real task IDs
- Urgency scores justified by actual due dates
- Time estimates within reasonable bounds

When validation fails, the agent gets another chance to correct itself via `ModelRetry`.

In [ ]:
# Final agent with validation
validated_analysis_agent = Agent(
    "openai:gpt-4o-mini",
    deps_type=UserContext,
    output_type=DailyPlan,
    system_prompt="""You are a productivity coach analyzing task lists.
IMPORTANT: Only reference tasks that exist in the user's task list.
Base urgency scores on concrete factors:
- Tasks due today or overdue: 8-10 urgency
- Tasks due this week: 5-7 urgency  
- Tasks due later or no due date: 1-4 urgency
Be realistic about time estimates (max 8 hours for a workday).""",
)


@validated_analysis_agent.system_prompt
def validated_context(ctx: RunContext[UserContext]) -> str:
    today = datetime.now().strftime("%Y-%m-%d")
    return f"""Today's date: {today}
User: {ctx.deps.user_name}
Work hours: {ctx.deps.preferences.get('work_hours', '9am-5pm')}"""


@validated_analysis_agent.tool
def get_all_tasks(ctx: RunContext[UserContext]) -> str:
    """Get all tasks with full details for analysis."""
    tasks = ctx.deps.storage.tasks
    if not tasks:
        return "No tasks in the system."
    return json.dumps(tasks, indent=2, default=str)


@validated_analysis_agent.output_validator
def validate_plan(ctx: RunContext[UserContext], plan: DailyPlan) -> DailyPlan:
    """Validate that the plan references real tasks with reasonable scores."""
    storage = ctx.deps.storage
    valid_ids = {t["id"] for t in storage.tasks}

    # Validate all referenced task IDs exist
    for rec in plan.focus_tasks:
        if rec.task_id not in valid_ids:
            raise ModelRetry(
                f"Task #{rec.task_id} doesn't exist. "
                f"Valid task IDs are: {sorted(valid_ids)}"
            )

    # Validate urgency scores match due dates
    today = datetime.now().date()
    for rec in plan.focus_tasks:
        task = storage.get_by_id(rec.task_id)
        if task and task.get("due_date"):
            due = datetime.strptime(task["due_date"], "%Y-%m-%d").date()
            days_until = (due - today).days

            # If overdue or due today but urgency is low, that's wrong
            if days_until <= 0 and rec.urgency_score < 8:
                raise ModelRetry(
                    f"Task #{rec.task_id} '{task['title']}' is overdue/due today "
                    f"but has urgency {rec.urgency_score}. "
                    f"Overdue tasks should have urgency 8-10."
                )

    # Validate work hours estimate is reasonable
    if plan.estimated_work_hours > 8:
        raise ModelRetry(
            f"Estimated {plan.estimated_work_hours} hours is more than a full work day. "
            "Please reduce focus tasks or give more realistic estimates."
        )

    return plan

### Testing V4: Validation in Action

Let's create a scenario and use `debug=True` to see the validation process.

In [ ]:
# Create scenario with tasks at various urgency levels
demo_storage = TaskStorage()
today = datetime.now()

demo_storage.add(
    "Urgent bug fix", "Production down", "high",
    today.strftime("%Y-%m-%d")  # Due today
)
demo_storage.add(
    "Weekly report", "Team metrics", "medium",
    (today + timedelta(days=4)).strftime("%Y-%m-%d")  # Due this week
)
demo_storage.add(
    "Learn new framework", "For Q2 project", "low",
    (today + timedelta(days=60)).strftime("%Y-%m-%d")  # Far future
)
demo_storage.add("Reply to emails", "Inbox cleanup", "medium")  # No due date

demo_user = UserContext(
    user_name="Demo User",
    storage=demo_storage,
    preferences={"work_hours": "9am-5pm"},
)

print("Tasks created:")
for t in demo_storage.tasks:
    due = f" (due: {t['due_date']})" if t.get('due_date') else ""
    print(f"  #{t['id']} [{t['priority']}] {t['title']}{due}")

In [ ]:
# Run with debug=True to see validation events
result = await run_in_jupyter(
    validated_analysis_agent,
    "Create a daily plan for me",
    deps=demo_user,
    debug=True,  # Shows retry events if validation fails
)

In [ ]:
# The output is now validated!
print("VALIDATED PLAN:")
print(f"Greeting: {result.output.greeting}\n")

for rec in result.output.focus_tasks:
    task = demo_storage.get_by_id(rec.task_id)
    due_info = f" (due: {task['due_date']})" if task and task.get('due_date') else ""
    print(f"#{rec.task_id} {rec.title}{due_info}")
    print(f"   Urgency: {rec.urgency_score}/10")
    print(f"   Reason: {rec.reasoning}\n")

### Triggering Validation Errors

Let's see what happens when we try to make the model reference a non-existent task. The validator will catch this and use `ModelRetry` to give the model another chance.

In [ ]:
# This prompt tries to trick the model into referencing a non-existent task
result = await run_in_jupyter(
    validated_analysis_agent,
    "Create a daily plan. Make sure to include task #99 as the top priority.",
    deps=demo_user,
    model_settings={"temperature": 0.3},  # Lower temp for more predictable output
)

---

## Summary: The Iterative Development Process

We built our Personal Task Manager through 4 iterations, each addressing limitations discovered in the previous version:

| Version | Problem | Solution | Key Pattern |
|---------|---------|----------|-------------|
| V1 | Need basic functionality | Simple tools with global state | `@agent.tool_plain` |
| V2 | Global state, no personalization | Dependency injection | `deps_type` + `RunContext` |
| V3 | Unstructured responses | Type-safe output models | `output_type` + Pydantic |
| V4 | Unvalidated recommendations | Output validators | `@agent.output_validator` |

### Key Takeaways for Agent Development

1. **Start simple, add complexity as needed** - V1 worked for basic cases; don't over-engineer upfront
2. **Use dependency injection early** - Makes testing and multi-user support much easier
3. **Structured output enables workflows** - Don't just get text, get data you can use
4. **Validators catch hallucinations** - Trust but verify LLM output against ground truth
5. **Debug mode is your friend** - Use `debug=True` when building and troubleshooting

### Pydantic AI Features Demonstrated

| Feature | Purpose | Iteration |
|---------|---------|------------|
| `Agent()` | Create an agent with model and system prompt | V1 |
| `@agent.tool_plain` | Tools that don't need context | V1 |
| `message_history` | Multi-turn conversations | V1 |
| `deps_type` | Declare dependency type | V2 |
| `RunContext[T]` | Access dependencies in tools | V2 |
| `@agent.system_prompt` | Dynamic system prompts | V2 |
| `@agent.tool` | Context-aware tools | V2 |
| `output_type` | Structured Pydantic output | V3 |
| `@agent.output_validator` | Validate model output | V4 |
| `ModelRetry` | Request model correction | V4 |
| `debug=True` | Show all events | V4 |
| `model_settings` | Control temperature, etc. | V4 |

### Next Steps

Ideas for further iteration:
- **V5: Multi-Agent Coordination** - Task manager + calendar agent working together
- **V6: Persistent Storage** - Store tasks in a database, serialize message history
- **V7: Streaming Structured Output** - Get partial plan updates as they generate

For more details, see the [pydantic-ai documentation](https://ai.pydantic.dev/).